### Overview
This notebook performs the initial data extraction and transformation of raw movie data from CSV format into a structured pandas DataFrame with proper data typing and organization.

In [ ]:
import csv
import pandas as pd

In [ ]:
path = "Elenco Movies definitivo.csv"

In [ ]:
# view CSV
with open(path, "r", encoding="UTF-8") as csv_file:
  csv_reader = csv.reader(csv_file, delimiter=",")

  header = next(csv_reader)
  print(f"Header: {header}")

  df = []
  for row in csv_reader:
    df.append(row)

In [ ]:
# create Film class
class Film:

  def __init__(self, movie_id, title, year, genres):

    self.movie_id = movie_id
    self.title = title
    self.year = year
    self.genres = genres


  def __str__(self):

    return f"Film: {self.movie_id}, {self.title}, {self.year}, {self.genres}"


In [ ]:
# Fix malformed CSV rows where quoted fields with embedded commas were incorrectly split
# Problem: CSV parser splits "Title, The (1995)" into multiple fields because of the comma
# Solution: Detect quote pairs and merge elements between opening and closing quotes

df_refined = []
for row in df:
  unified_string = None  # Temporary storage for quoted field being assembled
  new_row = []
  
  for elem in row:
    if '"' in elem:
      # Element contains a quote character
      if unified_string is None:
        # This is the opening quote - start collecting the quoted field
        unified_string = elem
      else:
        # This is the closing quote - merge with opening and add complete field to row
        new_row.append(unified_string + elem)
        unified_string = None  # Reset for next quoted field
    else:
      # Normal element without quotes - add directly to row
      new_row.append(elem)
  
  df_refined.append(new_row)

# Verify correction worked on a known problematic row
print(df_refined[72])

In [ ]:
def year_fetcher(row):
  """
  Extract movie title and year from combined title string.
  
  Handles edge cases where movie titles contain parentheses.
  Example: "Film (Part 1) (1995)" should split into "Film (Part 1)" and "1995"
  """
  # Split on opening parenthesis - last element should be the year
  temp = row[1].split("(")
  if len(temp) > 2:
    # Title contains parentheses (e.g., "Terminator 2: Judgment Day (Extended) (1991)")
    # Rejoin all parts except the last one (which is the year)
    title = "(".join([row for row in temp[:-1]])
    year = temp[-1]
  else:
    # Simple case: "Movie Title (1995)" splits into ["Movie Title ", "1995)"]
    title, year = temp

  return title, year

In [ ]:

for row in df_refined:
  try:
    # applying function
    title, year = year_fetcher(row)
    year = year.replace(")", "").strip()
    row[1] = title.strip()
    row.append(year)
  except:
    row.append(None)
# verify on sample
print(df_refined[72])

In [ ]:
# Convert pipe-delimited genre strings into lists
# Input format: "Action|Adventure|Sci-Fi" (single string)
# Output format: ["Action", "Adventure", "Sci-Fi"] (list)
for row in df_refined:
  genres = row[2].split("|")
  row[2] = genres

# verify on sample
print(df_refined[72])

In [ ]:
# Convert raw list data into Film objects for better structure and type safety
# Transforms from list format [id, title, genres, year] to Film class instances
df_films = []

for row in df_refined:
  # Extract fields from list positions
  movie_id = row[0]  # Unique identifier for the movie
  title = row[1]     # Cleaned movie title (without year)
  year = row[3]      # Release year (extracted earlier)
  genres = row[2]    # List of genre tags

  # Create Film object with extracted data
  film = Film(movie_id, title, year, genres)
  df_films.append(film)

# Verify  on sample
print(df_films[72])

In [ ]:
df_films[72].title

In [ ]:


# create an empty list to store values
data_for_pandas = []

# iterate over all films
for film in df_films:
    # transform object into simple dictionary
    film_dictionary = {
        "movie_id": film.movie_id,
        "title": film.title,
        "year": film.year,
        "genres": film.genres
    }
    data_for_pandas.append(film_dictionary)

# create the dataframe

df_final = pd.DataFrame(data_for_pandas)

# visualize first 5 rows
print(df_final.head())

In [ ]:
# make movie_id the index for now
df_final = df_final.set_index("movie_id")
print(df_final.head())

In [ ]:
pd.set_option('display.max_rows', 500)
print(df_final)

In [ ]:
# create parquet
df_final.to_parquet("Films.parquet")